# 05. Облачность, дождь, снег, град: обнаружение по бортовым датчикам и влияние на навигацию (угрозы №5, 7, 8, 9)

| № | Угроза | Эффект | Последствия |
|---|---|---|---|
| 5 | Облачность | нарушение оптики | потеря ориентации, вынужденная посадка |
| 7 | Осадки (дождь) | нарушение оптики | потеря ориентации, вынужденная посадка |
| 8 | Осадки (снег) | нарушение лидаров, рост веса | потеря ориентации, вынужденная посадка |
| 9 | Осадки (град) | физическое воздействие, рост веса | разрушение объекта, корректировка траектории |

В ноутбуке 2 осадки определялись по метеоданным (T, RH, ветер). Здесь - **по бортовым сенсорам в полёте** и с оценкой того, как они влияют на **навигацию**.

## Связь с ТЗ
| Сигнал | Источник | Как используем |
|---|---|---|
| видеопоток | вне потоков ТЗ (подтверждено) | частицы осадков, капли на линзе, потеря текстуры |
| LiDAR | подтверждено | помехи осадков в ближней зоне, фильтрация DROR |
| IMU (вертикальное ускорение) | БВС-7 (`component_status` IMU) / сырой IMU | удары града |
| `temperature`, `humidity` (→ T_w) | БВС-5 | различение дождя и снега |
| `wind_*`, `stabilization_status` | БВС-5 | контекст турбулентности |
| выход | БВС-3, НСУ-4/5 | флаги, доверие датчиков, выбор режима навигации |

## Что делаем
1. **Видео:** отличить дождь, снег и град по свойствам частиц (форма, размер, движение) и **не принять за них капли на линзе**; проверить пользу T_w.
2. **LiDAR:** убрать помехи осадков (DROR), не потеряв тонкие препятствия.
3. **Навигация:** сколько визуальных ориентиров переживает погода (прокси VIO на оптическом потоке) и какой режим навигации выбрать.
4. **Облачность в полёте:** отличить «внутри облака» от дымки и запотевания.
5. **Град:** ударные события по IMU и состояние «градовая обстановка».

> **Допущения:** синтетические сцены; движение камеры известно по телеметрии (1 px/кадр); осадки - случайные частицы без трекинга траекторий; LiDAR - угловая сетка с первым возвратом; реальных данных нет.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

from src.synth_weather import WxConfig, render_sequence, LidarConfig, lidar_scan, imu_series, detect_impacts
from src.weather_features import video_features, vio_inliers, dror, ror, sor, filter_metrics
from src.physics import wet_bulb_stull
plt.rcParams["figure.dpi"] = 110; pd.set_option("display.width", 200)

## 1. Симулятор
Видео: мир смещается на 1 px/кадр; атмосфера `I = J·T + A·(1−T)`; осадки - **частицы в воздухе** (полосы дождя, хлопья снега, крупные быстрые градины), в каждом кадре новые; **капли на линзе** - статичные размытые пятна (в координатах кадра). LiDAR: лучи на угловой сетке, помехи осадков заменяют первый возврат на малой дальности (дождь - низкая отражательная способность, снег - выше), часть помех образует скопления; препятствия (столб и «деревья») - угловые окна с меньшей дальностью.

In [ ]:
cases = [("ясно", {}), ("дождь 0.6", dict(rain=.6)), ("снег 0.6", dict(snow=.6)), ("град 0.7", dict(hail=.7)),
         ("капли на линзе", dict(drops=8)), ("запотевание 0.6", dict(lens_fog=.6)), ("дымка V=3км", dict(vis_m=3000)), ("облако V=100м", dict(vis_m=100))]
fig, axs = plt.subplots(1, 8, figsize=(16, 2.6))
for a, (n, kw) in zip(axs, cases):
    a.imshow(render_sequence(WxConfig(seed=3, n=4, **kw))[2]); a.set_title(n, fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## 2. Видео: тип осадков и устойчивость к каплям на линзе
**Признаки.** Движение камеры известно, поэтому разность `D_t = I_t(x) − I_{t−1}(x+1)` убирает неподвижный в мире фон; остаются частицы в воздухе и шум. По связным компонентам `|D_t| > 0,09`: число частиц, медианная и 90-й перцентиль площади, **вытянутость** (дождь - полосы), доля ориентации вдоль вертикали; **персистентность** `‖mean_t D‖² / mean_t ‖D_t‖²` (капли на линзе дают устойчивый остаток в координатах кадра, частицы - нет); энергия остатка, градиент и контраст кадра. Из БВС-5: температура мокрого термометра T_w.

**Классы:** `dry`, `rain`, `snow`, `hail`, `drops` (сухо, но капли на линзе), `rain+drops`. Baseline без обучения: порог на число частиц → вытянутость → площадь. Модель: градиентный бустинг. Для обучения и теста интенсивности осадков из разных диапазонов (смена условий).

In [ ]:
CL = ["dry", "rain", "snow", "hail", "drops", "rain+drops"]
def sample_tw(rng, cls):
    # допущение: T_w в БВС-5 коррелирует с типом осадков, но с перекрытием
    mu, sd = {"rain": (4, 3.5), "rain+drops": (4, 3.5), "snow": (-0.5, 1.5), "hail": (8, 3)}.get(cls, (6, 7))
    return rng.normal(mu, sd)
def make_cfg(rng, cls, lo, hi, seed):
    c = WxConfig(n=16, seed=seed, terrain=str(rng.choice(["veg", "mixed"])), alt=rng.uniform(150, 500), vis_m=rng.uniform(8000, 30000))
    I = rng.uniform(lo, hi)
    if cls in ("rain", "rain+drops"): c.rain = I
    if cls == "snow": c.snow = I
    if cls == "hail": c.hail = I
    if cls in ("drops", "rain+drops"): c.drops = int(rng.integers(4, 13))
    return c
def make_table(per, seed0, lo, hi, classes=CL):
    rng = np.random.default_rng(seed0); rows = []
    for cls in classes:
        for i in range(per):
            cfg = make_cfg(rng, cls, lo, hi, seed0 * 1000 + len(rows)); fr = render_sequence(cfg)
            f = video_features(fr); inl, tot = vio_inliers(fr, 5); f.update(vio_inl=inl, Tw=sample_tw(rng, cls), label=cls, I=cfg.rain + cfg.snow + cfg.hail)
            rows.append(f)
    return pd.DataFrame(rows)
train_v = make_table(40, 11, 0.3, 1.0)
test_v = make_table(30, 12, 0.3, 1.0)
test_light = make_table(25, 13, 0.1, 0.3)
FV = ["n_blobs", "area_med", "area_p90", "elong_med", "vert_med", "res_energy", "persist", "grad", "contrast", "mean"]
def rule(d):
    p = np.where(d.n_blobs < 2, "dry", np.where(d.elong_med > 3, "rain", np.where(d.area_p90 > 60, "hail", "snow")))
    return p
rows = []
for name, d in (("обычная интенсивность 0.3-1", test_v), ("слабая интенсивность 0.1-0.3", test_light)):
    for mname, feats in (("GBM видео", FV), ("GBM видео + T_w", FV + ["Tw"])):
        g = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(train_v[feats], train_v.label)
        rows.append(dict(тест=name, модель=mname, acc=(g.predict(d[feats]) == d.label).mean(), macroF1=f1_score(d.label, g.predict(d[feats]), average="macro")))
    p = rule(d); rows.append(dict(тест=name, модель="правило", acc=(p == d.label).mean(), macroF1=f1_score(d.label, p, average="macro")))
display(pd.DataFrame(rows).set_index(["тест", "модель"]).round(3))

In [ ]:
g_v = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(train_v[FV + ["Tw"]], train_v.label)
g_vo = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(train_v[FV], train_v.label)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
for a, (name, d, g, F) in zip(axs, (("видео", test_v, g_vo, FV), ("видео + T_w", test_v, g_v, FV + ["Tw"]))):
    cm = confusion_matrix(d.label, g.predict(d[F]), labels=CL, normalize="true")
    a.imshow(cm, vmin=0, vmax=1, cmap="Blues"); a.set_xticks(range(6), CL, rotation=45, ha="right", fontsize=8); a.set_yticks(range(6), CL, fontsize=8); a.set_title(name)
    for (i, j), v in np.ndenumerate(cm): a.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="k" if v < .6 else "w")
plt.tight_layout(); plt.show()
# кривая по интенсивности: обнаружение «идёт ли осадок» (rain/snow/hail/rain+drops против dry/drops) и признаки капель
rows = []
for I in (0.05, 0.1, 0.2, 0.3, 0.5, 1.0):
    d = make_table(15, 20 + int(I * 100), I, I, classes=["dry", "rain", "snow", "drops"])
    p = g_vo.predict(d[FV]); pos = d.label.isin(["rain", "snow"]); detp = np.isin(p, ["rain", "snow", "hail", "rain+drops"])
    rows.append(dict(I=I, recall_осадков=detp[pos].mean(), ложные_на_сухо_и_каплях=detp[~pos].mean()))
sev_v = pd.DataFrame(rows).set_index("I").round(2); display(sev_v)

## 3. LiDAR: помехи осадков и фильтрация
Помехи осадков - случайные возвраты на малой дальности. Сравниваются:
- **DROR** (Charron et al., 2018): радиус поиска растёт с дальностью, `r = max(r_min, k·R·Δθ)`, точка отбрасывается при числе соседей < `n_min`; применяется в ближней зоне (R < 30 м);
- **ROR**: радиус фиксирован (0,5 м);
- **SOR**: статистическая фильтрация по среднему расстоянию до k соседей.

Метрики: полнота и точность удаления помех, **сохранность препятствий** (включая тонкий столб в ближней зоне) и сохранность земли. Параметры DROR (k, n_min) подбираются на отдельных зёрнах.

In [ ]:
def eval_filters(kind, I, seeds, params):
    rows = []
    for s in seeds:
        sc = lidar_scan(LidarConfig(seed=s, **{kind: I}))
        for name, f in (("DROR", lambda sc: dror(sc["xyz"], sc["R"], k=params[0], n_min=params[1])), ("ROR", lambda sc: ror(sc["xyz"], sc["R"])), ("SOR", lambda sc: sor(sc["xyz"], sc["R"]))):
            m = filter_metrics(sc, f(sc)); m.update(метод=name, тип=kind, I=I); rows.append(m)
    return rows
tune = []
for k in (2, 3, 4, 6):
    for nm in (2, 3, 4):
        r = pd.DataFrame(sum([eval_filters(t, I, range(100, 106), (k, nm)) for t in ("rain", "snow") for I in (1.0, 2.0)], [])); r = r[r.метод == "DROR"]
        tune.append(dict(k=k, n_min=nm, clutter_recall=r.clutter_recall.mean(), obstacle_kept=r.obstacle_kept.mean(), ground_kept=r.ground_kept.mean()))
tune = pd.DataFrame(tune).round(3); display(tune.sort_values("clutter_recall", ascending=False).head(6))
ok = tune[(tune.obstacle_kept >= 0.99) & (tune.ground_kept >= 0.99)].sort_values("clutter_recall", ascending=False)
PAR = (int(ok.iloc[0].k), int(ok.iloc[0].n_min)) if len(ok) else (3, 3); print("выбраны параметры DROR (k, n_min) =", PAR)

In [ ]:
rows = sum([eval_filters(t, I, range(200, 208), PAR) for t in ("rain", "snow") for I in (0.25, 0.5, 1.0, 2.0)], [])
lf = pd.DataFrame(rows)
summ = lf.groupby(["тип", "I", "метод"])[["clutter_recall", "clutter_precision", "obstacle_kept", "ground_kept"]].mean().round(2)
display(summ.loc["rain"].unstack("метод")[["clutter_recall", "obstacle_kept"]]); display(summ.loc["snow"].unstack("метод")[["clutter_recall", "obstacle_kept"]])
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
for a, m in zip(axs, ("clutter_recall", "clutter_precision", "obstacle_kept")):
    for meth in ("DROR", "ROR", "SOR"):
        g = lf[lf.метод == meth].groupby("I")[m].mean(); a.plot(g.index, g.values, marker="o", label=meth)
    a.set_xlabel("интенсивность осадков"); a.set_title(m); a.grid(alpha=.3); a.set_ylim(0, 1.05)
axs[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
# интенсивность осадков по доле удалённых близких точек (без обучения) и влияние на число валидных возвратов
rows = []
for kind in ("rain", "snow"):
    for I in (0.0, 0.25, 0.5, 1.0, 2.0):
        for s in range(300, 306):
            sc = lidar_scan(LidarConfig(seed=s, **{kind: I})); k = dror(sc["xyz"], sc["R"], k=PAR[0], n_min=PAR[1])
            near = sc["R"] < 30; rows.append(dict(тип=kind, I=I, removed_frac=float((~k[near]).mean()), n_valid=int(k.sum()) / sc["n_rays"]))
ri = pd.DataFrame(rows)
for kind in ("rain", "snow"):
    d = ri[ri.тип == kind]; print(kind, "Spearman(доля удалённых, интенсивность) =", round(spearmanr(d.removed_frac, d.I)[0], 3))
display(ri.groupby(["тип", "I"]).mean().round(3))

## 4. Навигация: сколько ориентиров переживает погода
Прокси фронтенда визуальной одометрии: детектор углов Ши-Томаси + оптический поток Лукаса-Канаде вперёд-назад; **инлайер** - ошибка FB < 0,5 px и поток, согласованный с известным сдвигом. Для LiDAR - доля лучей с валидным возвратом после DROR. Режим навигации (правило): VIO при числе инлайеров ≥ 40; иначе LiDAR-одометрия при доле валидных ≥ 0,5; иначе инерциальная навигация + возврат (RTH).

In [ ]:
conds = [("ясно", {}, {}), ("дымка V=5 км", dict(vis_m=5000), dict(vis_m=5000)), ("дождь 0.5", dict(rain=.5), dict(rain=.5)), ("дождь 1.0", dict(rain=1.0), dict(rain=1.0)),
         ("снег 0.5", dict(snow=.5), dict(snow=.5)), ("снег 1.0", dict(snow=1.0), dict(snow=1.0)), ("капли на линзе", dict(drops=10), {}), ("запотевание 0.5", dict(lens_fog=.5), {}),
         ("запотевание 0.9", dict(lens_fog=.9), {}), ("град 0.7", dict(hail=.7), {}), ("облако V=300 м", dict(vis_m=300), dict(vis_m=300)), ("облако V=100 м", dict(vis_m=100), dict(vis_m=100))]
rows = []
for name, kv, kl in conds:
    for s in range(6):
        fr = render_sequence(WxConfig(n=10, seed=400 + s, **kv)); inl, tot = vio_inliers(fr, 6); f = video_features(fr)
        sc = lidar_scan(LidarConfig(seed=400 + s, **kl)); keep = dror(sc["xyz"], sc["R"], k=PAR[0], n_min=PAR[1])
        valid = float((keep & ~sc["clutter"]).sum()) / sc["n_rays"]
        mode = "VIO" if inl >= 40 else "LiDAR-одометрия" if valid >= 0.5 else "INS + RTH"
        rows.append(dict(условие=name, vio_inliers=inl, lidar_valid=valid, grad=f["grad"], contrast=f["contrast"], режим=mode))
nav = pd.DataFrame(rows)
tab = nav.groupby("условие", sort=False).agg(vio_inliers=("vio_inliers", "mean"), lidar_valid=("lidar_valid", "mean"), grad=("grad", "mean"), режим=("режим", lambda x: x.value_counts().index[0])).round(2)
display(tab)
fig, ax = plt.subplots(figsize=(9, 3.6)); x = np.arange(len(tab)); ax.bar(x - .2, tab.vio_inliers / 150, .4, label="VIO (инлайеры/150)"); ax.bar(x + .2, tab.lidar_valid, .4, label="LiDAR (доля валидных)")
ax.axhline(40 / 150, c="C0", ls=":"); ax.axhline(0.5, c="C1", ls=":"); ax.set_xticks(x, tab.index, rotation=35, ha="right", fontsize=8); ax.legend(fontsize=8); ax.set_ylim(0, 1.05); plt.tight_layout(); plt.show()

### Облачность в полёте (угроза №5)
Камера «внутри облака»: кадр почти однороден (низкий градиент и контраст), VIO теряет ориентиры. Из БВС-5: относительная влажность воздуха близка к 100 %. Проверяем различимость «облако» от остальных условий (дымка, запотевание, осадки, ясно) по видеопризнакам и по RH (допущение: в облаке RH 96-100 %, вне облака 30-95 %).

In [ ]:
rng = np.random.default_rng(55)
nav["cloud"] = nav.условие.str.startswith("облако")
nav["RH"] = np.where(nav.cloud, rng.uniform(96, 100, len(nav)), rng.uniform(30, 95, len(nav)))
res = {"градиент кадра": -nav.grad, "контраст кадра": -nav.contrast, "инлайеры VIO": -nav.vio_inliers, "RH (БВС-5)": nav.RH}
res["RH и градиент (min-ранг)"] = np.minimum(nav.RH.rank(pct=True), (-nav.grad).rank(pct=True))
display(pd.Series({k: roc_auc_score(nav.cloud, v) for k, v in res.items()}, name="AUROC «облако»").round(3).to_frame().T)
print("медианы по условиям - градиент:", nav.groupby("условие", sort=False).grad.median().round(3).to_dict())

## 5. Град: ударные события по IMU
Вертикальное ускорение 200 Гц: турбулентность (низкочастотная, порывы до 8·турб м/с²) + шум + гармоники ротора (120/240 Гц) + удары града (затухающие синусоиды 80 Гц, амплитуда 2-15 м/с²). Детектор: фильтр верхних частот 25 Гц, порог 6 робастных σ (MAD), рефрактерный период 40 мс. Состояние «градовая обстановка» - ≥ 3 ударов за 5 с. Подтверждение корреляцией с `precipitation_prob`/`turbulence_index` из НСУ-3 - вне этого ноутбука.

In [ ]:
def match(true_t, det_t, tol=0.025):
    det_used = set(); tp = 0
    for t in true_t:
        j = np.where(np.abs(det_t - t) <= tol)[0]
        if len(j): tp += 1; det_used.add(int(j[0]))
    return tp, len(det_t) - len(det_used)
rows = []
for rate in (0.0, 0.3, 1.0, 3.0, 8.0):
    for turb in (1.0, 3.0):
        for s in range(10):
            t, a, ev = imu_series(seed=800 + s, hail_rate=rate, turb=turb, tone=1.5)
            det, h, sig = detect_impacts(a); tp, fp = match(ev, det)
            storm = next((x for x in det if np.sum((det >= x - 5) & (det <= x)) >= 3), np.nan)
            rows.append(dict(rate=rate, turb=turb, n_true=len(ev), tp=tp, fp=fp, storm_t=storm, first_true=ev[2] if len(ev) >= 3 else np.nan))
ih = pd.DataFrame(rows)
agg = ih.groupby(["rate", "turb"]).apply(lambda g: pd.Series(dict(recall=g.tp.sum() / max(g.n_true.sum(), 1), ложных_ударов_на_запуск=g.fp.mean(),
          состояние_обнаружено=(g.storm_t.notna()).mean(), задержка_состояния_с=(g.storm_t - g.first_true).median())))
display(agg.round(2))
t, a, ev = imu_series(seed=801, hail_rate=3.0, turb=2.0, tone=1.5); det, h, sig = detect_impacts(a)
fig, axs = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
axs[0].plot(t, a, lw=.6); axs[0].set_ylabel("a_z, м/с²"); axs[1].plot(t, h, lw=.6); axs[1].set_ylabel("ВЧ (>25 Гц)")
axs[1].axhline(6 * sig, c="r", ls=":"); axs[1].axhline(-6 * sig, c="r", ls=":"); axs[1].plot(det, np.zeros(len(det)), "r|", ms=12, label="обнаружено"); axs[1].plot(ev, np.zeros(len(ev)) - 0.5, "k.", ms=4, label="истина")
axs[1].legend(fontsize=8); axs[1].set_xlabel("t, с"); plt.tight_layout(); plt.show()

## 6. Сводка и выход модуля
Выход - флаги угроз, доверие датчиков и рекомендуемый режим навигации. Все поля укладываются в БВС-3; значение `alert_type` для погоды в ТЗ отсутствует (согласовать).

In [ ]:
def weather_alert(t, flags, nav_mode, c_vio, c_lidar, uav_id="UAV-001"):
    sev = 4 if nav_mode == "INS + RTH" or "hail_storm" in flags else 3 if nav_mode != "VIO" else 2 if flags else 1
    act = {"VIO": "keep_nav; lower_rgb_weight_if_precip", "LiDAR-одометрия": "switch_nav_to_lidar_odometry; apply_DROR_filter",
           "INS + RTH": "activate_inertial_nav; return_to_home_or_land"}[nav_mode]
    if "hail_storm" in flags: act = "emergency_land_or_RTH; log_impact_energy"
    return {"alert_id": f"wx-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": sev, "description": "WX_" + "+".join(flags or ["NONE"]).upper(),
            "sensor_source": "video+lidar+imu", "confidence": {"vio": c_vio, "lidar": c_lidar}, "timestamp": int(t * 1000), "recommended_action": act}
weather_alert(42.0, ["rain", "lens_drops"], "LiDAR-одометрия", 0.31, 0.82)

## 8. Выводы и ограничения
(заполняется после запуска)